# M03 — CTA Data Quality Audit

M02 ki filtered bus aur rail CSVs ko check karna. Ye notebook sirf data read karti hai; files ko change ya clean nahi karti. Cells ko upar se neeche run karein.

## 1. Libraries aur data location

Notebook ko project root, `backend` ya `backend/notebooks` se run karein. CSVs `backend/data/processed` mein honi chahiye.

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

if Path("backend/data/processed").is_dir():
    data_folder = Path("backend/data/processed")      # project root
elif Path("data/processed").is_dir():
    data_folder = Path("data/processed")              # backend
else:
    data_folder = Path("../data/processed")            # backend/notebooks

print("Data folder:", data_folder.resolve())


Data folder: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\data\processed


## 2. Datasets load karo

Identifiers text mein read hote hain; khali ya sirf spaces wali values missing count hoti hain.

In [3]:
bus = pd.read_csv(data_folder / "bus_2022_2025.csv", dtype=str)
rail = pd.read_csv(data_folder / "rail_2022_2025.csv", dtype=str)

# Spaces hatana sirf notebook ke andar hai; CSV files nahi badalti.
bus = bus.apply(lambda column: column.str.strip().replace("", pd.NA))
rail = rail.apply(lambda column: column.str.strip().replace("", pd.NA))
datasets = {"Bus": bus, "Rail": rail}
print("Bus rows:", len(bus), "| Rail rows:", len(rail))


Bus rows: 167648 | Rail rows: 209472


## 3. Data overview

Columns aur pehli 5 rows dekh kar dataset ka structure samjho.

In [4]:
for name, df in datasets.items():
    print(f"\n{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("Columns:", list(df.columns))
    display(df.head())



Bus: 167,648 rows, 4 columns
Columns: ['route', 'date', 'daytype', 'rides']


,route,date,daytype,rides
0,1,01/03/2022,W,385
1,1,01/04/2022,W,470
2,1,01/05/2022,W,289
3,1,01/06/2022,W,339
4,1,01/07/2022,W,233



Rail: 209,472 rows, 5 columns
Columns: ['station_id', 'stationname', 'date', 'daytype', 'rides']


,station_id,stationname,date,daytype,rides
0,40010,Austin-Forest Park,01/01/2022,A,231
1,40020,Harlem-Lake,01/01/2022,A,500
2,40030,Pulaski-Lake,01/01/2022,A,284
3,40040,Quincy/Wells,01/01/2022,A,493
4,40050,Davis,01/01/2022,A,502


## 4. Missing values aur duplicate rows

Exact duplicate ka matlab: poori row repeat hui hai. Pehli row ke baad wali repeated rows count hoti hain.

In [5]:
for name, df in datasets.items():
    print(f"\n{name} — missing values per column")
    display(df.isna().sum().to_frame("Missing values"))
    print("Exact duplicate extra rows:", df.duplicated().sum())



Bus — missing values per column


,Missing values
route,0
date,0
daytype,0
rides,0


Exact duplicate extra rows: 0

Rail — missing values per column


,Missing values
station_id,0
stationname,0
date,0
daytype,0
rides,0


Exact duplicate extra rows: 0


## 5. Dates check karo

Dates 2022–2025 mein honi chahiye. Unreadable aur blank dates alag count hoti hain. Ye pandas ki date conversion hai, CSV mein koi change nahi hota.

In [6]:
dates = {}
for name, df in datasets.items():
    dates[name] = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    invalid = df["date"].notna() & dates[name].isna()
    outside = dates[name].notna() & ~dates[name].between("2022-01-01", "2025-12-31 23:59:59.999999999")
    print(f"\n{name} date range:", dates[name].min(), "to", dates[name].max())
    print("Invalid nonblank dates:", invalid.sum())
    print("Dates outside 2022–2025:", outside.sum())



Bus date range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00
Invalid nonblank dates: 0
Dates outside 2022–2025: 0

Rail date range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00
Invalid nonblank dates: 0
Dates outside 2022–2025: 0


## 6. Route/station aur date ki duplicates

Same bus route ya rail station ke liye ek hi date dobara ho to extra row count hoti hai. Missing ya invalid date aur missing ID wali rows is check mein shamil nahi hotin; unka count alag diya hai. Duplicate milne par pehle investigate karna hai, automatically delete nahi karna.

In [7]:
for name, df in datasets.items():
    id_column = "route" if name == "Bus" else "station_id"
    keys = pd.DataFrame({"id": df[id_column], "date": dates[name].dt.normalize()})
    valid = keys["id"].notna() & keys["date"].notna()
    print(f"{name}: duplicate ID + date extra rows:", keys[valid].duplicated().sum())
    print(f"{name}: rows excluded from this check:", (~valid).sum())


Bus: duplicate ID + date extra rows: 0
Bus: rows excluded from this check: 0
Rail: duplicate ID + date extra rows: 0
Rail: rows excluded from this check: 0


## 7. Ridership aur day types

Blank rides missing values mein hain. Non-numeric, negative aur fractional rides ko review karna chahiye. Zero rides yaad rahe: automatically error nahi. Expected day types `W`, `A` aur `U` hain.

In [8]:
for name, df in datasets.items():
    rides = pd.to_numeric(df["rides"], errors="coerce").replace([float("inf"), float("-inf")], float("nan"))
    invalid = df["rides"].notna() & rides.isna()
    unexpected = df["daytype"].notna() & ~df["daytype"].isin(["W", "A", "U"])
    print(f"\n{name}")
    print("Invalid nonblank rides:", invalid.sum())
    print("Negative rides:", (rides < 0).sum())
    print("Non-integer rides:", (rides.notna() & (rides % 1 != 0)).sum())
    print("Zero rides:", (rides == 0).sum())
    print("Unexpected nonblank day types:", unexpected.sum())
    display(df["daytype"].value_counts(dropna=False).to_frame("Rows"))



Bus
Invalid nonblank rides: 124270
Negative rides: 0
Non-integer rides: 0
Zero rides: 2
Unexpected nonblank day types: 0


,Rows
daytype,
W,127701
U,20222
A,19725



Rail
Invalid nonblank rides: 125516
Negative rides: 0
Non-integer rides: 0
Zero rides: 2742
Unexpected nonblank day types: 0


,Rows
daytype,
W,146243
U,33264
A,29965


## 8. Conclusions

Apne outputs ke basis par note karein: missing values, duplicates ya invalid values hain? Zero rides aur missing route/station days ko automatically errors na samjhein. Data ko clean karne ka decision aglay phase mein hoga. Ye historical data hai, live monitoring nahi.